# Acceptance: custom posterior with partial pooling

**This is a sign-off page, not a tutorial.** It exercises roadmap step 46
(composability package `CP3`, findings `CX21`, `CX22`, `CX35`). The workflow
stages accept any `Posterior`; here the main epidemic stays a `BayesianModel`,
while a hand-written numpyro model adds partially pooled historical outbreaks
that share the recovery rate. The main data only pin the early growth rate, so
`infection` and `recovery` form a ridge until the side durations inform
`recovery`. Nothing here asserts; the automated checks live in
`tests/test_calib_posterior.py::test_gate_*`.

Run every cell top to bottom (about two minutes). Each section ends with a
**What to check** box.

| | You are checking that… |
|---|---|
| 1 | main-data log density has a diagonal ridge and the truth sits on it |
| 2 | `NumpyroPosterior(bm.numpyro_model())` matches `bm` on `log_density` to ~1e-10 |
| 3 | the joint object lists sites, shapes, and LHS vs Predictive fallback |
| 4 | with the same Adam budget, main-only ends collapse to one wrong ridge point; joint ends sit within ±0.05 of the true recovery |
| 5 | gray main-only NUTS draws lie along the ridge; joint draws are tight (10–90% width under 30% of main-only) with the truth inside the joint 10–90% interval |


In [ ]:
import warnings

import jax
import jax.numpy as jnp
import numpy as np
import numpyro
import numpyro.distributions as dist
import optax
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from numpyro.infer import MCMC, NUTS

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

from summer4 import (
    Compartments,
    FlowModel,
    Param,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Target,
    TargetSet,
    TransitionFlow,
)
from summer4.epi import FOIKind, ForceOfInfection, MixingMatrix
from summer4.epi.calibration import BayesianModel, NegativeBinomial, NumpyroPosterior, Uniform
from summer4.epi.calibration import workflow as wf


## Build the model and synthetic data

A frequency-dependent SIR on one million people. The main epidemic is fit to
noisy infectious counts every two days over **days 0–20**. Six historical
outbreaks each contribute **30** observed infectious durations from
`Exponential(recovery_j)` with `recovery_j = exp(mu + tau * z_j)` under a
plate. The main epidemic uses the same hierarchy:
`recovery = exp(mu + tau * z_main)`. Hyperpriors on `mu` and `tau` are **vague**
(`mu ~ Normal(log(0.25), 1)` — not centred on the generating value) and
`tau ~ HalfNormal(0.15)` for tight pooling. The joint model scores the main
targets through `bm.log_likelihood` inside `numpyro.factor`. The cells below fix
every seed and setting; stars mark the generating parameters (`DATA_SEED = 7`).


In [ ]:
# --- shared settings (kept in sync with tests/gate_custom_posterior.py) ---
DATA_SEED = 7
K_OUTBREAKS = 6
N_DUR = 30
TRUE_INFECTION = 0.45
TRUE_MU = -1.1
TRUE_TAU = 0.1
MU_PRIOR_LOC = float(np.log(0.25))
MU_PRIOR_SCALE = 1.0
TAU_PRIOR_SCALE = 0.15
NB_DISPERSION = 20.0
POP_SIZE = 1.0e6
OPT_SEED = 0
LHS_MAIN_SEED = 0
LHS_JOINT_SEED = 0
MCMC_MAIN_SEED = 3
MCMC_JOINT_SEED = 4
MCMC_SAMPLES_PER_CHAIN = 200
MCMC_CHAINS = 2
MCMC_WARMUP_START = 40
MCMC_WARMUP_MIN = 60
OPT_MAX_STEPS = 800
OPT_CHUNK = 50
OPT_LR = 0.05
LHS_M_MAIN = 48
LHS_M_JOINT = 48
N_OPT_STARTS = 8

rng = np.random.default_rng(DATA_SEED)
z_hist = rng.normal(size=K_OUTBREAKS)
z_main = float(rng.normal())
TRUE_RECOVERY = float(np.exp(TRUE_MU + TRUE_TAU * z_main))
rates_hist = np.exp(TRUE_MU + TRUE_TAU * z_hist)
durations = rng.exponential(1.0 / rates_hist[:, None], size=(K_OUTBREAKS, N_DUR))

state = Property("state", ("S", "I", "R"))
pop = Property("pop", ("all",))
pmap = PropertyMap.from_property(state).stratify(pop)
flow_model = FlowModel(pmap)
flow_model.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        ForceOfInfection(
            "infection",
            infectious=state["I"],
            group_by=pop,
            mixing=MixingMatrix(pop, np.array([[1.0]])),
            kind=FOIKind.FREQUENCY,
            contact_rate=Param("infection"),
        ),
    )
)
flow_model.add_flow(TransitionFlow("recovery", state["I"], state["R"], Param("recovery")))
cm = flow_model.compile()

y0_values = np.zeros(pmap.size)
y0_values[pmap.select(state["S"])] = POP_SIZE - 20.0
y0_values[pmap.select(state["I"])] = 20.0
y0 = PropertyData.wrap(pmap, y0_values)
infectious = Compartments(where=state["I"])
run_kwargs = {"t0": 0.0, "t1": 180.0, "dt": 0.5, "solver": "euler"}
times = np.arange(0.0, 21.0, 2.0)

clean = cm.run(
    {"infection": TRUE_INFECTION, "recovery": TRUE_RECOVERY},
    y0,
    save=SavePlan(requests={"I": SaveRequest(infectious, ts=times)}),
    **run_kwargs,
)
expected = np.asarray(clean["I"].values.data).reshape(-1)
observed = rng.negative_binomial(20, 20 / (20 + expected)).astype(float)

bm = BayesianModel(
    cm,
    {},
    priors=(Uniform("infection", 0.1, 1.5), Uniform("recovery", 0.02, 1.0)),
    targets=TargetSet(
        targets=(
            Target(
                key="I",
                times=times,
                values=observed,
                quantity=infectious,
                likelihood=NegativeBinomial(dispersion=NB_DISPERSION),
            ),
        )
    ),
    y0=y0,
    run_kwargs=run_kwargs,
)

truth = {
    "infection": TRUE_INFECTION,
    "recovery": TRUE_RECOVERY,
    "mu": TRUE_MU,
    "tau": TRUE_TAU,
    "z_main": z_main,
}


def joint_model() -> None:
    mu = numpyro.sample("mu", dist.Normal(MU_PRIOR_LOC, MU_PRIOR_SCALE))
    tau = numpyro.sample("tau", dist.HalfNormal(TAU_PRIOR_SCALE))
    infection = numpyro.sample("infection", dist.Uniform(0.1, 1.5))
    with numpyro.plate("outbreaks", K_OUTBREAKS):
        z = numpyro.sample("z", dist.Normal(0.0, 1.0))
        rate = jnp.exp(mu + tau * z)
        with numpyro.plate("dur", N_DUR):
            numpyro.sample(
                "duration",
                dist.Exponential(rate),
                obs=jnp.asarray(durations.T),
            )
    z_main_s = numpyro.sample("z_main", dist.Normal(0.0, 1.0))
    recovery = jnp.exp(mu + tau * z_main_s)
    numpyro.factor(
        "main",
        bm.log_likelihood({"infection": infection, "recovery": recovery}),
    )


joint_post = NumpyroPosterior(joint_model)
wrapped_bm = NumpyroPosterior(bm.numpyro_model())


def recovery_from_joint_params(params):
    mu = np.asarray(params["mu"], float)
    tau = np.asarray(params["tau"], float)
    z_main_arr = np.asarray(params["z_main"], float)
    return np.exp(mu + tau * z_main_arr)

pd.Series(
    {
        "data seed": DATA_SEED,
        "true infection": TRUE_INFECTION,
        "true recovery (main)": TRUE_RECOVERY,
        "generating mu": TRUE_MU,
        "generating tau": TRUE_TAU,
        "mu prior": f"Normal({MU_PRIOR_LOC:.3f}, {MU_PRIOR_SCALE})",
        "tau prior": f"HalfNormal({TAU_PRIOR_SCALE})",
        "durations per outbreak": N_DUR,
    },
    name="setup",
)


## 1. The ridge in main-data log density

Contour the main-only `bm` log density and mark the truth.

In [ ]:
inf_grid = np.linspace(0.15, 1.2, 28)
rec_grid = np.linspace(0.05, 0.95, 28)
C, R = np.meshgrid(inf_grid, rec_grid)
surface = wf.evaluate(
    bm,
    wf.Candidates.from_params(bm, {"infection": C.ravel(), "recovery": R.ravel()}),
    batch_size=200,
)
log_density = np.asarray(surface.log_density).reshape(C.shape)
best = float(log_density.max())

fig = go.Figure(
    go.Contour(
        x=inf_grid,
        y=rec_grid,
        z=np.clip(log_density, best - 25, None),
        colorscale="Viridis",
        contours={"coloring": "heatmap", "showlines": False},
        colorbar={"title": "log density"},
    )
)
fig.add_scatter(
    x=[TRUE_INFECTION],
    y=[TRUE_RECOVERY],
    mode="markers",
    marker={"symbol": "star", "size": 16, "color": "gold", "line": {"width": 1}},
    name="truth",
)
fig.update_layout(
    title="1. Main-data log density: ridge between infection and recovery",
    xaxis_title="infection rate",
    yaxis_title="recovery rate",
    height=560,
)
fig.show()


> **What to check**
>
> - A bright diagonal band runs across the panel, not a single isolated peak.
> - The gold star lies on that band.

## 2. Same density two ways

Compare `bm` and `NumpyroPosterior(bm.numpyro_model())` on `log_density` (x64).

In [ ]:
rng_pts = np.random.default_rng(0)
sites = bm.prior_names()
pts = [{name: float(rng_pts.normal()) for name in sites} for _ in range(24)]

ld_bm = []
ld_wrap = []
# jax 0.11+: jax.enable_x64; jax 0.6: jax.experimental.enable_x64
with (
    jax.enable_x64(True)
    if hasattr(jax, "enable_x64")
    else jax.experimental.enable_x64()
):
    for z in pts:
        ld_bm.append(float(bm.log_density(z)))
        ld_wrap.append(float(wrapped_bm.log_density(z)))

diff = np.abs(np.asarray(ld_bm) - np.asarray(ld_wrap))
fig = pd.DataFrame({"bm": ld_bm, "NumpyroPosterior": ld_wrap}).plot(
    kind="scatter",
    x="bm",
    y="NumpyroPosterior",
    title="2. log_density: bm vs NumpyroPosterior(bm.numpyro_model())",
)
fig.add_trace(
    go.Scatter(
        x=[min(ld_bm), max(ld_bm)],
        y=[min(ld_bm), max(ld_bm)],
        mode="lines",
        name="y = x",
        line={"dash": "dash", "color": "gray"},
    )
)
fig.update_layout(xaxis_title="bm log_density", yaxis_title="wrapped log_density", height=520)
fig.show()

pd.Series({"max |difference|": float(diff.max())}, name="2")


> **What to check**
>
> - Points sit on the dashed `y = x` line.
> - `max |difference|` is about `1e-10` or smaller.

## 3. Joint posterior object

Site names, shapes, and LHS vs Predictive fallback.

In [ ]:
lhs_joint = wf.lhs(joint_post, 8, seed=2)
settings = lhs_joint.history[0].settings

pd.DataFrame(
    {
        "prior_names": [joint_post.prior_names()],
        "site_shapes": [joint_post.site_shapes()],
        "LHS stratifies": [settings["lhs_sites"]],
        "Predictive fills": [settings["predictive_sites"]],
        "site_priors (names)": [tuple(p.name for p in joint_post.site_priors())],
    }
).T


> **What to check**
>
> - `prior_names` lists `mu`, `tau`, `infection`, `z`, `z_main` (plate member `z` has shape `(K,)`).
> - `LHS stratifies` includes `mu`, `tau`, `infection`, and `z_main`; `Predictive fills` is `["z"]` only.

## 4. LHS → evaluate → multi-start optimise

Main-only and joint share the same LHS size, best(`N_OPT_STARTS`) starts, Adam
learning rate (`OPT_LR`), chunk size and step budget. The surface is flat along
lines of constant `infection − recovery`, the quantity the early data pin, so
likelihood alone does not pick a unique optimum on the ridge. With a long Adam
run every main-only start can still collapse to one wrong point: the optimiser
maximises log density in unconstrained space, which includes the log-Jacobian of
the `Uniform` priors' logit transforms, and that term peaks inside the box — the
end point is set by the priors, not the data. The duration data let the joint
model pin `recovery` near the truth.


In [ ]:
design_main = wf.evaluate(
    bm, wf.lhs(bm, LHS_M_MAIN, seed=LHS_MAIN_SEED), batch_size=LHS_M_MAIN
)
starts_main = design_main.best(N_OPT_STARTS)
opt_method = wf.Optax(optax.adam(OPT_LR))
opt_main = wf.optimize(
    bm,
    starts_main,
    method=opt_method,
    chunk_steps=OPT_CHUNK,
    max_steps=OPT_MAX_STEPS,
    seed=OPT_SEED,
)
ends_main = opt_main.candidates

design_joint = wf.evaluate(
    joint_post, wf.lhs(joint_post, LHS_M_JOINT, seed=LHS_JOINT_SEED), batch_size=16
)
starts_joint = design_joint.best(N_OPT_STARTS)
opt_joint = wf.optimize(
    joint_post,
    starts_joint,
    method=opt_method,
    chunk_steps=OPT_CHUNK,
    max_steps=OPT_MAX_STEPS,
    seed=OPT_SEED,
)
ends_joint = opt_joint.candidates

inf_main = np.asarray(ends_main.params["infection"], float)
rec_main = np.asarray(ends_main.params["recovery"], float)
inf_joint = np.asarray(ends_joint.params["infection"], float)
rec_joint = recovery_from_joint_params(ends_joint.params)

fig = go.Figure()
fig.add_scatter(
    x=inf_main,
    y=rec_main,
    mode="markers+text",
    text=[str(i) for i in range(len(inf_main))],
    textposition="top center",
    marker={"size": 10, "symbol": "circle-open"},
    name="main-only ends",
)
fig.add_scatter(
    x=inf_joint,
    y=rec_joint,
    mode="markers",
    marker={"size": 12, "symbol": "x", "color": "crimson"},
    name="joint ends",
)
fig.add_scatter(
    x=[TRUE_INFECTION],
    y=[TRUE_RECOVERY],
    mode="markers",
    marker={"symbol": "star", "size": 16, "color": "gold"},
    name="truth",
)
fig.update_layout(
    title="4. Multi-start optimiser end points",
    xaxis_title="infection rate",
    yaxis_title="recovery rate",
    height=560,
)
fig.show()

spread = pd.DataFrame(
    {
        "main recovery mean": [float(rec_main.mean())],
        "main recovery ptp": [float(np.ptp(rec_main))],
        "joint recovery mean": [float(rec_joint.mean())],
        "joint recovery ptp": [float(np.ptp(rec_joint))],
        "|main mean - truth|": [float(abs(rec_main.mean() - TRUE_RECOVERY))],
        "max |joint - truth|": [float(np.max(np.abs(rec_joint - TRUE_RECOVERY)))],
    }
)
spread


> **What to check**
>
> - Open circles sit on top of each other (`main recovery ptp` under **0.01**) at recovery about **0.57**, far from the truth (**0.33**).
> - Red crosses cluster near the truth (`max |joint - truth|` under **0.05**).

## 5. Staged NUTS: ridge vs partial pooling

`wf.warmup_until` then `wf.sample_until`, seeded from `Candidates.init_params`.
**200** post-warmup draws per chain, **2** chains.

In [ ]:
warm_rule = wf.WarmupRule(min_warmup=MCMC_WARMUP_MIN, rhat=1.15)
stop = wf.StopRule(rhat=None, ess=None, max_samples=MCMC_SAMPLES_PER_CHAIN)


def make_mcmc_main(num_warmup: int, num_chains: int = MCMC_CHAINS) -> MCMC:
    return MCMC(
        NUTS(bm.numpyro_model(), target_accept_prob=0.9),
        num_warmup=num_warmup,
        num_samples=MCMC_SAMPLES_PER_CHAIN,
        num_chains=num_chains,
    )


def make_mcmc_joint(num_warmup: int, num_chains: int = MCMC_CHAINS) -> MCMC:
    return MCMC(
        NUTS(joint_post.numpyro_model(), target_accept_prob=0.9),
        num_warmup=num_warmup,
        num_samples=MCMC_SAMPLES_PER_CHAIN,
        num_chains=num_chains,
    )


init_main = design_main.init_params(MCMC_CHAINS, seed=MCMC_MAIN_SEED)
warm_main = wf.warmup_until(
    make_mcmc_main, MCMC_WARMUP_START, warm_rule, init_params=init_main, seed=MCMC_MAIN_SEED
)
run_main = wf.sample_until(warm_main.mcmc, stop, seed=MCMC_MAIN_SEED)

init_joint = design_joint.init_params(MCMC_CHAINS, seed=MCMC_JOINT_SEED)
warm_joint = wf.warmup_until(
    make_mcmc_joint, MCMC_WARMUP_START, warm_rule, init_params=init_joint, seed=MCMC_JOINT_SEED
)
run_joint = wf.sample_until(warm_joint.mcmc, stop, seed=MCMC_JOINT_SEED)

inf_draws_main = run_main.samples["infection"].reshape(-1)
rec_draws_main = run_main.samples["recovery"].reshape(-1)
mu_draws = run_joint.samples["mu"].reshape(-1)
tau_draws = run_joint.samples["tau"].reshape(-1)
zm_draws = run_joint.samples["z_main"].reshape(-1)
inf_draws_joint = run_joint.samples["infection"].reshape(-1)
rec_draws_joint = np.exp(mu_draws + tau_draws * zm_draws)

fig = go.Figure()
fig.add_scatter(
    x=inf_draws_main,
    y=rec_draws_main,
    mode="markers",
    marker={"size": 5, "opacity": 0.45, "color": "gray"},
    name="main-only NUTS",
)
fig.add_scatter(
    x=inf_draws_joint,
    y=rec_draws_joint,
    mode="markers",
    marker={"size": 5, "opacity": 0.5, "color": "crimson"},
    name="joint NUTS",
)
fig.add_scatter(
    x=[TRUE_INFECTION],
    y=[TRUE_RECOVERY],
    mode="markers",
    marker={"symbol": "star", "size": 16, "color": "gold"},
    name="truth",
)
fig.update_layout(
    title="5. NUTS draws in (infection, recovery)",
    xaxis_title="infection rate",
    yaxis_title="recovery rate",
    height=560,
)
fig.show()

width_main = np.quantile(rec_draws_main, 0.9) - np.quantile(rec_draws_main, 0.1)
width_joint = np.quantile(rec_draws_joint, 0.9) - np.quantile(rec_draws_joint, 0.1)
q_joint = np.quantile(rec_draws_joint, [0.1, 0.5, 0.9])

z_draws = run_joint.samples["z"]
mu_by = run_joint.samples["mu"][:, :, None]
tau_by = run_joint.samples["tau"][:, :, None]
recovery_by_outbreak = np.exp(mu_by + tau_by * z_draws)
main_recovery_draws = rec_draws_joint

fig2 = go.Figure()
for j in range(K_OUTBREAKS):
    fig2.add_box(y=recovery_by_outbreak[:, :, j].reshape(-1), name=f"outbreak {j}")
fig2.add_box(y=main_recovery_draws, name="main epidemic", marker_color="gold")
fig2.add_hline(y=TRUE_RECOVERY, line_dash="dash", annotation_text="true main recovery")
fig2.update_layout(
    title="5. Partial pooling: per-outbreak recovery vs main epidemic",
    yaxis_title="recovery rate",
    height=520,
    showlegend=False,
)
fig2.show()

pd.DataFrame(
    {
        "main 10–90% width": [float(width_main)],
        "joint 10–90% width": [float(width_joint)],
        "width ratio (joint / main)": [float(width_joint / width_main)],
        "joint 10%": [float(q_joint[0])],
        "joint 50%": [float(q_joint[1])],
        "joint 90%": [float(q_joint[2])],
        "truth in joint 10–90%": [bool(q_joint[0] <= TRUE_RECOVERY <= q_joint[2])],
    }
)


> **What to check**
>
> - **Gray** main-only draws form a loose diagonal swath; crimson joint draws sit in a tight blob.
> - `width ratio (joint / main)` is about **0.14** (under **0.3**); `truth in joint 10–90%` is **True**.
> - The pooling figure shows outbreak boxes partially shrunk toward the main epidemic; the dashed line crosses the main box near its median.